# RAG with Feast Feature Store — Milvus + PostgreSQL + Ray

This notebook demonstrates a **Retrieval Augmented Generation (RAG)** pipeline using:

| Component | Technology |
|-----------|------------|
| **Data Source** | HuggingFace `rajpurkar/squad` via `RaySource` |
| **Compute Engine** | Ray (distributed embedding generation via KubeRay) |
| **Online Store** | Milvus (vector similarity search) |
| **Offline Store** | PostgreSQL |
| **Registry** | PostgreSQL (SQL) |

### How it works
1. `RaySource` reads the SQuAD dataset directly from HuggingFace via `ray.data.from_huggingface()`
2. The `BatchFeatureView` UDF runs on Ray workers to generate embeddings with `sentence-transformers`
3. `store.materialize()` pushes embeddings into Milvus for vector search
4. `retrieve_online_documents_v2` performs semantic similarity queries

### Prerequisites
- Feast FeatureStore CR deployed and `Ready` (via `setup.sh`)
- Milvus, PostgreSQL, and Ray cluster running in the namespace
- This notebook runs **inside the cluster** (e.g. OpenShift AI workbench) with network access to the Feast services

## 1. Install Dependencies

In [ ]:
%pip install --quiet feast[milvus,ray] sentence-transformers psycopg2-binary "pyarrow>=14.0.0"

## 2. Read Feature Store Configuration

Load `feature_store.yaml` to inspect how the Feast project is configured — stores, registry, compute engine, and data source.

In [ ]:
import yaml
from pathlib import Path

FS_YAML_PATH = "/opt/app-root/src/feast-config/rag_demo"
fs_yaml_path = Path(FS_YAML_PATH)

with open(fs_yaml_path) as f:
    fs_config = yaml.safe_load(f)

print("=" * 50)
print("  Feast Feature Store Configuration")
print("=" * 50)
for key, value in fs_config.items():
    if isinstance(value, dict):
        print(f"\n  {key}:")
        for k, v in value.items():
            print(f"    {k}: {v}")
    else:
        print(f"  {key}: {value}")
print("=" * 50)

## 3. Connect to Feature Store & List Registered Features

In [ ]:
from feast import FeatureStore

store = FeatureStore(fs_yaml_file=FS_YAML_PATH)

print(f"Project: {store.project}")
print(f"\nEntities ({len(store.list_entities())}):")
for entity in store.list_entities():
    print(f"  - {entity.name} (join_keys: {entity.join_keys})")

print(f"\nFeature Views ({len(store.list_all_feature_views())}):")
for fv in store.list_all_feature_views():
    fv_type = type(fv).__name__
    print(f"  - {fv.name} ({fv_type})")
    if hasattr(fv, 'source'):
        print(f"      source: {type(fv.source).__name__} — {fv.source.name}")
    for field in fv.schema:
        tags = ""
        if hasattr(field, 'vector_index') and field.vector_index:
            tags = f" [vector, dim={field.vector_length}]"
        print(f"      {field.name}: {field.dtype}{tags}")

print(f"\nFeature Services ({len(store.list_feature_services())}):")
for fs in store.list_feature_services():
    print(f"  - {fs.name} (tags: {fs.tags})")

## 4. Materialize — Ray reads HuggingFace, generates embeddings, writes to Milvus

This single call triggers the full pipeline:
1. **RaySource** reads `rajpurkar/squad` from HuggingFace via `ray.data.from_huggingface()`
2. **Ray UDF** (`PassageEmbeddingProcessor`) generates embeddings distributedly on Ray workers using `sentence-transformers/all-MiniLM-L6-v2`
3. **Milvus** receives the embeddings for vector similarity search

In [ ]:
from datetime import datetime, timezone

start = datetime(2020, 1, 1, tzinfo=timezone.utc)
end = datetime(2026, 12, 31, 23, 59, 59, tzinfo=timezone.utc)

print("Starting materialization...")
print("  RaySource → HuggingFace rajpurkar/squad")
print("  Ray UDF   → sentence-transformers embedding generation")
print("  Milvus    → vector store write")
print()

store.materialize(start_date=start, end_date=end)

print("\nMaterialization complete! Embeddings are in Milvus.")

## 5. RAG Retrieval — Query Similar Passages

Encode a natural-language query into an embedding, then use Feast's `retrieve_online_documents_v2` to find the most similar passages from Milvus.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

queries = [
    "What is the capital of France?",
    "How does photosynthesis work in plants?",
    "Who wrote the theory of relativity?",
    "What are the largest cities in Europe?",
]

for query in queries:
    query_embedding = model.encode([query], normalize_embeddings=True)[0].tolist()

    results = store.retrieve_online_documents_v2(
        features=[
            "passage_embeddings:embedding",
            "passage_embeddings:title",
            "passage_embeddings:context",
        ],
        query=query_embedding,
        top_k=3,
    )

    df_results = results.to_df()

    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")

    if not df_results.empty:
        for i, row in df_results.iterrows():
            title = row.get("title", "N/A")
            context = row.get("context", "")
            distance = row.get("distance", "N/A")
            snippet = context[:200] + "..." if len(str(context)) > 200 else context
            print(f"\n  [{i+1}] Title: {title}  |  Score: {distance}")
            print(f"      {snippet}")
    else:
        print("  No results found.")

## Architecture

```
┌──────────────────────────────────────────────────────┐
│                    RAG Pipeline                       │
├──────────────────────────────────────────────────────┤
│                                                      │
│  HuggingFace SQuAD ──► RaySource                     │
│       │                 (reader_type=huggingface)     │
│       ▼                                              │
│  Ray Cluster (KubeRay)                               │
│  ┌─────────────────────────────────┐                 │
│  │ BatchFeatureView (mode=ray)     │                 │
│  │ UDF: PassageEmbeddingProcessor  │                 │
│  │   sentence-transformers         │                 │
│  │   all-MiniLM-L6-v2             │                 │
│  └───────────┬─────────────────────┘                 │
│              │                                       │
│       ┌──────┴──────┐                                │
│       ▼             ▼                                │
│  PostgreSQL      Milvus                              │
│  (offline +      (online store)                      │
│   registry)      vector search                       │
│                     │                                │
│                     ▼                                │
│          retrieve_online_documents_v2                │
│          (semantic similarity search)                │
│                     │                                │
│                     ▼                                │
│             Top-K Passages ──► LLM (optional)        │
└──────────────────────────────────────────────────────┘
```